In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from tqdm import tqdm
import seaborn
from functools import reduce
from operator import getitem
from analysis.color_palettes import make_alternate_versions as make_color_palettes
seaborn.set_style('whitegrid')

# Always run from repo root regardless of notebook location
_REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
if os.path.basename(os.getcwd()) == "analysis":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from utils.model_registry import get_token_count

In [ ]:
FINEWEB_PADDED = "rankme_fineweb_padded"
# TRAINSET_PACKED_UNC_ONLY = "rankme_trainset_packed_unc"
TRAINSET_PACKED = "rankme_trainset_packed_4MT"
# FULL_LIMITED_5K = "full_limited_5k"
# KFAC_ARXIV = "kfac_small_arxiv"
KFAC_SMALL_SHUFFLED = "kfac_small_shuffled"
BLOCK_REPR = "block_representations"

In [ ]:
def load_results(model_name: str, dataset_name: str = 'fineweb'):
    new_path = os.path.join('data', 'results', dataset_name, f'results_{model_name}.npy')
    old_path = os.path.join('data', 'results', f'results_{model_name}.npy')
    path = new_path if os.path.exists(new_path) else old_path
    try:
        res = np.load(path, allow_pickle=True).item()
    except FileNotFoundError as e:
        return None, None
    step_nums = sorted(res.keys())
    # print(res)
    return res, step_nums

In [ ]:
import json
import re

def dictable(d):
    try:
        dict(d)
        return True
    except (TypeError, ValueError):
        return False


def deleave(d):
    return { k: deleave(dict(v)) if dictable(v) else type(v).__name__ for k,v in d.items()}


def one_layer_one_head(checkpoint, layer=None, head=0):
    """Trim a checkpoint dict to a single block (default: first present) and, within
    its attention, a single head — so the dump shows one example of each node kind
    instead of every head in every layer. Non-block nodes (residual) are kept."""
    blk = re.compile(r'blk(\d+)')
    hd  = re.compile(r'blk\d+\.attn\.head(\d+)')
    blks = sorted({int(m.group(1)) for k in checkpoint if (m := blk.match(k))})
    keep = blks[0] if (layer is None and blks) else layer
    out = {}
    for k, v in checkpoint.items():
        m = blk.match(k)
        if not m:
            out[k] = v
            continue
        if int(m.group(1)) != keep:
            continue
        hm = hd.match(k)
        if hm and int(hm.group(1)) != head:
            continue
        out[k] = v
    return out


def info_group(
    data_sources: list[str],
    model_names: list[str],
    layer=None,
    head=0,
):
    for data_source in data_sources:
        pythia_passed = 0
        olmo_passed = 0
        print("data_source:", data_source)
        for model_name in tqdm(model_names):
            res, step_nums = load_results(model_name, dataset_name=data_source)
            if res is None:
                continue
            if 'pythia' in model_name:
                if pythia_passed:
                    continue
                pythia_passed = 1
            if 'OLMo' in model_name:
                if olmo_passed:
                    continue
                olmo_passed = 1
            final_checkpoint = one_layer_one_head(res[step_nums[-1]], layer, head)

            print("model:", model_name)
            print(json.dumps(deleave(final_checkpoint), indent=2).replace('"',''))

            if olmo_passed and pythia_passed:
                break


In [ ]:
model_names = [
    # 'OLMo-2-0425-1B',
    'OLMo-2-1124-7B',
    # 'pythia-1b-deduped',
    'pythia-6.9b-deduped',
    # 'pythia-70m-deduped',
    # 'pythia-31m-deduped',
    # 'pythia-14m-deduped',
]

data_sources = [
    TRAINSET_PACKED,
    FINEWEB_PADDED,
    KFAC_SMALL_SHUFFLED,
    BLOCK_REPR,
]


In [ ]:
info_group(data_sources[-1:], model_names)

In [ ]:
info_group(data_sources[:-1], model_names)